# Program Record Builder

## Script
Single row DataFrame → Excel

## Purpose
Create a single program record and export it

## What it does
- Builds a dictionary with program details
- Converts dictionary to DataFrame
- Exports to Excel (pima_programs.xlsx)

## Key Notes
- Assumes variables already exist
- Used for testing or manual entry

In [5]:
import pandas as pd

data = {
    "Program": title,
    "Degree": degree,
    "CIP": cip[0] if cip else None,
    "Credits": credits[0] if credits else None,
    "Courses": ", ".join(courses[:10])
}

df = pd.DataFrame([data])

df.to_excel("pima_programs.xlsx", index=False)

print(df)

           Program Degree      CIP Credits  \
0  Accounting, AAS    AAS  52.0301      61   

                                             Courses  
0  WRT 154   - recommended) 3 Credits, BUS 151   ...  


# Pima Catalog Web Scraper

## Script
requests + BeautifulSoup scraper

## Purpose
Scrape all Pima programs and build base dataset

## What it does
- Requests catalog page and extracts program links
- Visits each program page
- Extracts program title, CIP, credits, duration, degree, and courses
- Combines results into a DataFrame
- Saves to PIMA_AI_Programs_FULL.xlsx

## Key Notes
- This is the starting dataset for the pipeline

In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

# ---------------------------
# STEP 1: Get all program links
# ---------------------------
base_url = "https://catalog.pima.edu/"
catalog_url = base_url + "content.php?catoid=13&navoid=2066"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(catalog_url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

links = soup.find_all("a")

program_links = []
for link in links:
    href = link.get("href")
    if href and "preview_program.php" in href:
        full_link = base_url + href
        program_links.append(full_link)

program_links = list(set(program_links))  # remove duplicates

print(f"Found {len(program_links)} program links")

# ---------------------------
# STEP 2: Scrape each program
# ---------------------------
data = []

for url in program_links:
    try:
        page = requests.get(url, headers=headers)
        soup = BeautifulSoup(page.text, "html.parser")
        text = soup.get_text()

        # Program Name
        title_tag = soup.find("h1")
        if not title_tag:
            continue
        title = title_tag.text.strip()

        # CIP
        cip_match = re.findall(r"CIP Code:\s*(\d+\.\d+)", text)
        cip = cip_match[0] if cip_match else None

        # Credits
        credits_match = re.findall(r"Program Total:\s*(\d+)", text)
        credits = credits_match[0] if credits_match else None

        # ✅ Duration (correctly placed)
        duration_match = re.findall(r"Duration in weeks:\s*(\d+)", text)
        duration = duration_match[0] if duration_match else None

        # Degree Type
        if "AAS" in title:
            degree = "AAS"
        elif "Certificate" in title:
            degree = "Certificate"
        elif "AA" in title or "AS" in title:
            degree = "Associate"
        else:
            degree = "Other"

        # Courses (first few)
        courses = re.findall(r"[A-Z]{3}\s?\d{3}.*?Credits", text)
        courses_clean = ", ".join(courses[:8])

        data.append({
            "Program": title,
            "Degree": degree,
            "CIP": cip,
            "Credits": credits,
            "Duration_Weeks": duration,  # ✅ included
            "Courses": courses_clean,
            "URL": url
        })

        print(f"Scraped: {title}")

    except Exception as e:
        print(f"Error with {url}: {e}")

# ---------------------------
# STEP 3: Create DataFrame
# ---------------------------
df = pd.DataFrame(data)

if df.empty:
    print("❌ No data collected — check scraping step")
else:
    print("✅ Data collected successfully")

    # Clean numeric columns
    df["Credits"] = pd.to_numeric(df["Credits"], errors="coerce")
    df["Duration_Weeks"] = pd.to_numeric(df["Duration_Weeks"], errors="coerce")

    # ---------------------------
    # STEP 4: Save
    # ---------------------------
    df.to_excel("PIMA_AI_Programs_FULL.xlsx", index=False)

    print("📁 File saved: PIMA_AI_Programs_FULL.xlsx")
    print(df.head(10))

Found 102 program links
Scraped: Veterinary Service Specialist Certificate
Scraped: Pharmacy Technology Certificate
Scraped: Medical Laboratory Technician, AAS
Scraped: Clinical Research Professional, Post Degree Certificate
Scraped: Plumber Certificate
Scraped: Welding, Basic Certificate
Scraped: Fire Science Academy Track Certificate
Scraped: Education, Special Education Endorsement for Certified Teachers Post-Baccalaureate Certificate
Scraped: County Corrections Training Academy Certificate
Scraped: Cabinetmaker Certificate
Scraped: Respiratory Care, AAS
Scraped: Automotive Mechanics Certificate
Scraped: Digital Film Arts and Animation, Digital and Film Arts Concentration, AAS
Scraped: Digital Arts, Design Concentration, AAS
Scraped: Class A Vehicle Driver Certificate
Scraped: Class B Commercial Driver’s License Certificate
Scraped: Practical Nurse Certificate
Scraped: Accounting, AAS
Scraped: Digital Film Arts and Animation, Digital Animation Concentration, AAS
Scraped: Social Serv

# CIP to SOC Mapping

## Script
Merge with CIP2020_SOC2018_Crosswalk.xlsx

## Purpose
Map programs (CIP) to occupations (SOC)

## What it does
- Loads program dataset
- Loads CIP-SOC crosswalk
- Renames columns to align fields
- Cleans CIP formatting
- Merges on CIP
- Calculates match rate
- Saves to PIMA_with_SOC.xlsx

## Key Notes
- CIP formatting must match exactly

In [18]:
import pandas as pd

# ---------------------------
# STEP 1: Load your Pima program dataset
# ---------------------------
df = pd.read_excel("PIMA_AI_Programs_FULL.xlsx")

print("✅ Loaded program dataset")
print(df.columns)

# ---------------------------
# STEP 2: Load CIP → SOC crosswalk (correct sheet)
# ---------------------------
crosswalk_path = r"C:\Users\301533\Documents\AI Analysis\CIP2020_SOC2018_Crosswalk.xlsx"

cip_soc_crosswalk = pd.read_excel(
    crosswalk_path,
    sheet_name="CIP-SOC"
)

print("✅ Loaded crosswalk")
print(cip_soc_crosswalk.columns)

# ---------------------------
# STEP 3: Rename columns (CRITICAL)
# ---------------------------
cip_soc_crosswalk = cip_soc_crosswalk.rename(columns={
    "CIP2020Code": "CIP",
    "SOC2018Code": "SOC"
})

# ---------------------------
# STEP 4: Clean CIP format (MOST IMPORTANT STEP)
# ---------------------------

# Convert to string
df["CIP"] = df["CIP"].astype(str).str.strip()
cip_soc_crosswalk["CIP"] = cip_soc_crosswalk["CIP"].astype(str).str.strip()

# Fix formatting (ensure 6-digit CIP like 52.0301)
df["CIP"] = df["CIP"].str.zfill(7)
cip_soc_crosswalk["CIP"] = cip_soc_crosswalk["CIP"].str.zfill(7)

# ---------------------------
# STEP 5: Merge
# ---------------------------
df_merged = df.merge(
    cip_soc_crosswalk[["CIP", "SOC"]],
    on="CIP",
    how="left"
)

print("✅ Merge complete")

# ---------------------------
# STEP 6: Check results
# ---------------------------
print(df_merged[["Program", "CIP", "SOC"]].head(10))

# How many matched?
match_rate = df_merged["SOC"].notna().mean()
print(f"✅ Match rate: {round(match_rate * 100, 2)}%")

# ---------------------------
# STEP 7: Save
# ---------------------------
df_merged.to_excel("PIMA_with_SOC.xlsx", index=False)

print("📁 Saved: PIMA_with_SOC.xlsx")

✅ Loaded program dataset
Index(['Program', 'Degree', 'CIP', 'Credits', 'Duration_Weeks', 'Courses',
       'URL'],
      dtype='object')
✅ Loaded crosswalk
Index(['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title'], dtype='object')
✅ Merge complete
                                             Program      CIP      SOC
0          Veterinary Service Specialist Certificate  01.8203  43-1011
1          Veterinary Service Specialist Certificate  01.8203  43-4171
2                    Pharmacy Technology Certificate  51.0805  25-1071
3                    Pharmacy Technology Certificate  51.0805  29-2052
4                 Medical Laboratory Technician, AAS  51.1004  25-1071
5                 Medical Laboratory Technician, AAS  51.1004  29-2012
6  Clinical Research Professional, Post Degree Ce...  51.0719  11-9033
7  Clinical Research Professional, Post Degree Ce...  51.0719  11-9111
8  Clinical Research Professional, Post Degree Ce...  51.0719  11-9199
9                              

# AI Exposure Integration

## Script
Merge with AI PEW dataset and scoring

## Purpose
Add AI exposure and compute program-level AI risk

## What it does
- Loads program + SOC dataset
- Loads AI dataset
- Cleans SOC formatting
- Merges AI data
- Converts AI category to numeric score
- Aggregates to program-level score
- Converts score back to Low, Medium, High
- Saves to PIMA_with_AI_EXPOSURE.xlsx

## Key Notes
- Produces SOC-level and program-level AI exposure

In [19]:
import pandas as pd

# ---------------------------
# STEP 1: Load your merged program file (WITH SOC)
# ---------------------------
df = pd.read_excel("PIMA_with_SOC.xlsx")

print("✅ Loaded program dataset")
print(df.columns)

# ---------------------------
# STEP 2: Load AI exposure dataset
# ---------------------------
ai_path = r"C:\Users\301533\Documents\AI Analysis\AI PEW 08182025.xlsx"

ai = pd.read_excel(ai_path)

print("✅ Loaded AI dataset")
print(ai.columns)

# ---------------------------
# STEP 3: Clean SOC format (CRITICAL)
# ---------------------------

# Ensure both match format like 43-1011
df["SOC"] = df["SOC"].astype(str).str.strip()
ai["SOC Code"] = ai["SOC Code"].astype(str).str.strip()

# ---------------------------
# STEP 4: Select only needed AI columns
# ---------------------------

ai_subset = ai[[
    "SOC Code",
    "Category",      # 🔥 THIS is your AI exposure category
    "ILO category",  # optional but useful
    "AI_0",
    "AI_1",
    "Tj"
]]

# Rename for merge
ai_subset = ai_subset.rename(columns={
    "SOC Code": "SOC",
    "Category": "AI_Category"
})

# ---------------------------
# STEP 5: Merge AI exposure onto your programs
# ---------------------------

df = df.merge(ai_subset, on="SOC", how="left")

print("✅ AI merge complete")

# ---------------------------
# STEP 6: CHECK RESULTS
# ---------------------------

print(df[[
    "Program",
    "CIP",
    "SOC",
    "AI_Category"
]].head(20))

match_rate = df["AI_Category"].notna().mean()
print(f"✅ AI match rate: {round(match_rate * 100, 2)}%")

# ---------------------------
# STEP 7: CREATE PROGRAM-LEVEL AI SCORE
# ---------------------------

# Convert categories to numeric scale
ai_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

df["AI_Score"] = df["AI_Category"].map(ai_map)

# Aggregate to PROGRAM level
program_ai = df.groupby("Program").agg({
    "AI_Score": "mean"
}).reset_index()

# Optional: convert back to labels
def score_to_label(x):
    if pd.isna(x):
        return None
    elif x < 1.5:
        return "Low"
    elif x < 2.5:
        return "Medium"
    else:
        return "High"

program_ai["Program_AI_Category"] = program_ai["AI_Score"].apply(score_to_label)

# ---------------------------
# STEP 8: MERGE BACK TO PROGRAM FILE
# ---------------------------

df = df.merge(program_ai, on="Program", how="left")

# ---------------------------
# STEP 9: SAVE FINAL OUTPUT
# ---------------------------

df.to_excel("PIMA_with_AI_EXPOSURE.xlsx", index=False)

print("📁 Saved: PIMA_with_AI_EXPOSURE.xlsx")

✅ Loaded program dataset
Index(['Program', 'Degree', 'CIP', 'Credits', 'Duration_Weeks', 'Courses',
       'URL', 'SOC'],
      dtype='object')
✅ Loaded AI dataset
Index(['Merged_SOC_Code', 'area', 'Area Name', 'SOC Code', 'SOC Title',
       'SOC Level', '2024 Estimate', '2026 Projection',
       '2 Year Numeric Change', '2 Year Percent Change', '2 Year Exits',
       'Annual Exits', '2 Year Transfers', 'Annual Transfers', 'Due to Growth',
       'Annual Growth', '2 Year Total Openings', 'Annual Total Openings',
       'EducationValue', 'WorkExperienceValue', 'JobTrainingValue',
       'ConfidentialFlag', 'GrowthOpenings', 'AnnualGrowthOpenings',
       'ReplacementOpenings', 'AnnualReplacementOpenings',
       'TotalReplacementOpenings', 'AnnualTotalReplacementOpenings',
       'GrowthRate', 'firsttwo', 'Occupational Group', 'Annual Percent Change',
       'AI_0', 'AI_1', 'Tj', 'SjL', 'SjM', 'SjH', 'rank_SjL', 'rank_SjH',
       'Category', 'ILO category', 'ILO Mean'],
      dtype='o

# Course Parsing and Data Restructuring

## Script
Explode and regex cleaning

## Purpose
Convert course strings into structured tables

## What it does
- Cleans SOC and AI fields
- Splits course strings into lists
- Explodes into one row per course
- Extracts course code and name
- Creates program-SOC-AI table
- Creates program-course table
- Saves multiple outputs

## Key Notes
- Converts messy text into relational format

In [27]:
import pandas as pd
import datetime

# =====================================================
# STEP 1: LOAD DATA
# =====================================================
df = pd.read_excel("PIMA_with_AI_EXPOSURE_latest.xlsx")

print("✅ Loaded data")
print(df.columns)

# =====================================================
# STEP 2: CLEAN DATA
# =====================================================

# Clean SOC
df["SOC"] = df["SOC"].astype(str).str.strip()

# Clean AI Category
df["AI_Category"] = df["AI_Category"].fillna("Unknown")

# Fix ILO formatting
df["ILO category"] = (
    df["ILO category"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("  ", " ", regex=False)
    .str.strip()
)

df["ILO category"] = df["ILO category"].replace("nan", pd.NA)

# Fix AI score column names (only if they exist)
df = df.rename(columns={
    "AI_Score_x": "SOC_AI_Score",
    "AI_Score_y": "Program_AI_Score"
})

# =====================================================
# STEP 3: HANDLE COURSES (SAFE)
# =====================================================

# If Course_List DOES NOT exist → create it
if "Course_List" not in df.columns:
    print("⚠️ Course_List not found — creating it from Courses")

    df["Course_List"] = df["Courses"].astype(str).str.split(",")

    df = df.explode("Course_List")

# Clean Course_List
df["Course_List"] = df["Course_List"].astype(str).str.strip()

# Keep FULL original text
df["Course_Full"] = df["Course_List"]

# Extract Course Code (BIO 156)
df["Course_Code"] = df["Course_List"].str.extract(r'([A-Z]{2,4}\s?\d{3})')

# Extract Course Name (text AFTER dash)
df["Course_Name"] = df["Course_List"].str.split("-").str[1]

# Clean Course Name
df["Course_Name"] = df["Course_Name"].str.replace(r'\d+\s*Credits?', '', regex=True)
df["Course_Name"] = df["Course_Name"].str.strip()

# Fallback if no dash exists
df["Course_Name"] = df["Course_Name"].fillna(df["Course_List"])

# =====================================================
# STEP 4: CREATE CLEAN TABLES
# =====================================================

# Table 1: Program-SOC-AI
df_program_soc = df.drop_duplicates(subset=["Program", "SOC"])

# Table 2: Program-Courses
df_program_courses = df[["Program", "Course_Code"]].drop_duplicates()

# Table 3: Full Long
df_full_long = df.copy()

# =====================================================
# STEP 5: SAVE FILES
# =====================================================
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

file1 = f"PIMA_PROGRAM_SOC_AI_{timestamp}.xlsx"
file2 = f"PIMA_PROGRAM_COURSES_{timestamp}.xlsx"
file3 = f"PIMA_FULL_LONG_{timestamp}.xlsx"

df_program_soc.to_excel(file1, index=False)
df_program_courses.to_excel(file2, index=False)
df_full_long.to_excel(file3, index=False)

# Latest versions
df_program_soc.to_excel("PIMA_PROGRAM_SOC_AI_latest.xlsx", index=False)
df_program_courses.to_excel("PIMA_PROGRAM_COURSES_latest.xlsx", index=False)
df_full_long.to_excel("PIMA_FULL_LONG_latest.xlsx", index=False)

# =====================================================
# STEP 6: SUMMARY
# =====================================================
print("\n📁 Files saved:")
print(file1)
print(file2)
print(file3)

print("\n📊 Summary:")
print(f"Programs: {df['Program'].nunique()}")
print(f"SOCs: {df['SOC'].nunique()}")
print(f"Courses: {df['Course_Code'].nunique()}")

print("✅ DONE — NO MORE ERRORS")

✅ Loaded data
Index(['Program', 'Degree', 'CIP', 'Credits', 'Duration_Weeks', 'Courses',
       'URL', 'SOC', 'AI_Category', 'ILO category', 'AI_0', 'AI_1', 'Tj',
       'SOC_AI_Score', 'Program_AI_Score', 'Program_AI_Category'],
      dtype='object')
⚠️ Course_List not found — creating it from Courses

📁 Files saved:
PIMA_PROGRAM_SOC_AI_20260318_1320.xlsx
PIMA_PROGRAM_COURSES_20260318_1320.xlsx
PIMA_FULL_LONG_20260318_1320.xlsx

📊 Summary:
Programs: 102
SOCs: 197
Courses: 370
✅ DONE — NO MORE ERRORS


# Course Name Lookup Scraper

## Script
Course page scraper from URLs

## Purpose
Replace messy course names with clean titles

## What it does
- Loads program URLs
- Finds course links
- Scrapes each course page
- Extracts clean course titles
- Builds lookup table
- Merges clean names back
- Overwrites main dataset

## Key Notes
- Improves data quality significantly

In [28]:
import pandas as pd
import datetime

# =====================================================
# STEP 1: LOAD DATA
# =====================================================
df = pd.read_excel("PIMA_with_AI_EXPOSURE_latest.xlsx")

print("✅ Loaded data")
print(df.columns)

# =====================================================
# STEP 2: CLEAN DATA
# =====================================================

# Clean SOC
df["SOC"] = df["SOC"].astype(str).str.strip()

# Clean AI Category
df["AI_Category"] = df["AI_Category"].fillna("Unknown")

# Fix ILO formatting
df["ILO category"] = (
    df["ILO category"]
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("  ", " ", regex=False)
    .str.strip()
)

df["ILO category"] = df["ILO category"].replace("nan", pd.NA)

# Fix AI score column names (only if they exist)
df = df.rename(columns={
    "AI_Score_x": "SOC_AI_Score",
    "AI_Score_y": "Program_AI_Score"
})

# =====================================================
# STEP 3: HANDLE COURSES (SAFE)
# =====================================================

# If Course_List DOES NOT exist → create it
if "Course_List" not in df.columns:
    print("⚠️ Course_List not found — creating it from Courses")

    df["Course_List"] = df["Courses"].astype(str).str.split(",")

    df = df.explode("Course_List")

# Clean Course_List
df["Course_List"] = df["Course_List"].astype(str).str.strip()

# Keep FULL original text
df["Course_Full"] = df["Course_List"]

# Extract Course Code (BIO 156)
df["Course_Code"] = df["Course_List"].str.extract(r'([A-Z]{2,4}\s?\d{3})')

# Extract Course Name (text AFTER dash)
df["Course_Name"] = df["Course_List"].str.split("-").str[1]

# Clean Course Name
df["Course_Name"] = df["Course_Name"].str.replace(r'\d+\s*Credits?', '', regex=True)
df["Course_Name"] = df["Course_Name"].str.strip()

# Fallback if no dash exists
df["Course_Name"] = df["Course_Name"].fillna(df["Course_List"])

# =====================================================
# STEP 4: CREATE CLEAN TABLES
# =====================================================

# Table 1: Program-SOC-AI
df_program_soc = df.drop_duplicates(subset=["Program", "SOC"])

# Table 2: Program-Courses
df_program_courses = df[["Program", "Course_Code"]].drop_duplicates()

# Table 3: Full Long
df_full_long = df.copy()

# =====================================================
# STEP 5: SAVE FILES
# =====================================================
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

file1 = f"PIMA_PROGRAM_SOC_AI_{timestamp}.xlsx"
file2 = f"PIMA_PROGRAM_COURSES_{timestamp}.xlsx"
file3 = f"PIMA_FULL_LONG_{timestamp}.xlsx"

df_program_soc.to_excel(file1, index=False)
df_program_courses.to_excel(file2, index=False)
df_full_long.to_excel(file3, index=False)

# Latest versions
df_program_soc.to_excel("PIMA_PROGRAM_SOC_AI_latest.xlsx", index=False)
df_program_courses.to_excel("PIMA_PROGRAM_COURSES_latest.xlsx", index=False)
df_full_long.to_excel("PIMA_FULL_LONG_latest.xlsx", index=False)

# =====================================================
# STEP 6: SUMMARY
# =====================================================
print("\n📁 Files saved:")
print(file1)
print(file2)
print(file3)

print("\n📊 Summary:")
print(f"Programs: {df['Program'].nunique()}")
print(f"SOCs: {df['SOC'].nunique()}")
print(f"Courses: {df['Course_Code'].nunique()}")

print("✅ DONE — NO MORE ERRORS")

✅ Loaded data
Index(['Program', 'Degree', 'CIP', 'Credits', 'Duration_Weeks', 'Courses',
       'URL', 'SOC', 'AI_Category', 'ILO category', 'AI_0', 'AI_1', 'Tj',
       'SOC_AI_Score', 'Program_AI_Score', 'Program_AI_Category'],
      dtype='object')
⚠️ Course_List not found — creating it from Courses

📁 Files saved:
PIMA_PROGRAM_SOC_AI_20260318_1656.xlsx
PIMA_PROGRAM_COURSES_20260318_1656.xlsx
PIMA_FULL_LONG_20260318_1656.xlsx

📊 Summary:
Programs: 102
SOCs: 197
Courses: 370
✅ DONE — NO MORE ERRORS


# Course Name Lookup Scraper

## Script
Web scraper that extracts clean course names from program pages and updates the main dataset

## Purpose
Standardize and improve course names by pulling official titles directly from course pages

## What it does

### Loads program dataset
- Reads PIMA_FULL_LONG_latest.xlsx
- Extracts unique program URLs from the URL column
- Sets up request headers for web scraping

### Loops through program pages
- Sends request to each program URL
- Parses HTML using BeautifulSoup
- Finds all links on the page

### Identifies course links
- Filters links where text matches course code pattern:
  - Format like BIO 156 or AVM 202
- Converts relative URLs to full URLs using urljoin

### Prevents duplicate scraping
- Tracks visited course URLs using a set
- Skips already processed course pages

### Scrapes course pages
- Requests each course page
- Extracts page title (from h1 if available)
- Uses regex to split:
  - Course_Code
  - Course_Name
- Cleans extra whitespace from course names

### Handles unmatched formats
- If regex does not match:
  - Keeps course code from link text
  - Leaves course name as null

### Builds lookup table
- Stores:
  - Program_URL
  - Course_URL
  - Course_Code
  - Course_Name
- Converts results into DataFrame
- Removes duplicate rows

### Saves lookup file
- Exports to PIMA_COURSE_LOOKUP_FROM_LINKS.xlsx
- Prints preview of data

### Cleans course codes in main dataset
- Extracts standardized course codes using regex
- Removes extra whitespace

### Creates simplified lookup
- Keeps one record per Course_Code
- Sorts and removes duplicates

### Merges lookup back to main dataset
- Left joins on Course_Code
- Adds cleaned Course_Name values

### Replaces messy course names
- Uses cleaned lookup name if available
- Falls back to original name if needed

### Saves updated dataset
- Overwrites PIMA_FULL_LONG_latest.xlsx
- Ensures Tableau uses cleaned data

## Key Notes
- Regex is critical for identifying valid course codes
- Duplicate tracking improves performance
- Lookup merge standardizes course names across programs
- This step significantly improves data quality for Tableau

In [6]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import urljoin

# ==========================================
# STEP 1: LOAD YOUR FILE WITH PROGRAM URLS
# ==========================================
df = pd.read_excel("PIMA_FULL_LONG_latest.xlsx")

program_urls = df["URL"].dropna().unique()

headers = {
    "User-Agent": "Mozilla/5.0"
}

results = []
seen_course_urls = set()

# ==========================================
# STEP 2: LOOP THROUGH PROGRAM PAGES
# ==========================================
for program_url in program_urls:
    try:
        r = requests.get(program_url, headers=headers, timeout=15)
        soup = BeautifulSoup(r.text, "html.parser")

        # Find all links on the page
        links = soup.find_all("a", href=True)

        for link in links:
            href = link["href"]
            text = link.get_text(" ", strip=True)

            # Only keep links that look like actual course links
            # and whose visible text looks like a course code
            if re.match(r"^[A-Z]{2,4}\s?\d{3}[A-Z]{0,2}$", text):
                course_url = urljoin(program_url, href)

                # Avoid scraping the same course page repeatedly
                if course_url in seen_course_urls:
                    continue
                seen_course_urls.add(course_url)

                try:
                    cr = requests.get(course_url, headers=headers, timeout=15)
                    csoup = BeautifulSoup(cr.text, "html.parser")

                    # Usually the course page title is in h1
                    h1 = csoup.find("h1")
                    if h1:
                        title_text = h1.get_text(" ", strip=True)
                    else:
                        title_text = csoup.get_text(" ", strip=True)

                    # Example expected:
                    # "AVM 202 - Aviation Safety"
                    m = re.search(r"([A-Z]{2,4}\s?\d{3}[A-Z]{0,2})\s*-\s*(.+)", title_text)

                    if m:
                        course_code = m.group(1).strip()
                        course_name = m.group(2).strip()

                        # Clean a few common leftovers
                        course_name = re.sub(r"\s+", " ", course_name).strip()

                        results.append({
                            "Program_URL": program_url,
                            "Course_URL": course_url,
                            "Course_Code": course_code,
                            "Course_Name": course_name
                        })

                        print(f"✔ {course_code} -> {course_name}")
                    else:
                        results.append({
                            "Program_URL": program_url,
                            "Course_URL": course_url,
                            "Course_Code": text.strip(),
                            "Course_Name": None
                        })
                        print(f"⚠ {text.strip()} -> no title match")

                    time.sleep(0.3)

                except Exception as e:
                    print(f"❌ Course page error: {course_url} -> {e}")

        time.sleep(0.3)

    except Exception as e:
        print(f"❌ Program page error: {program_url} -> {e}")

# ==========================================
# STEP 3: BUILD LOOKUP TABLE
# ==========================================
course_lookup = pd.DataFrame(results)

# Drop duplicates by course URL or code/name combo
course_lookup = course_lookup.drop_duplicates(subset=["Course_URL", "Course_Code", "Course_Name"])

# Save lookup
course_lookup.to_excel("PIMA_COURSE_LOOKUP_FROM_LINKS.xlsx", index=False)

print("✅ Saved: PIMA_COURSE_LOOKUP_FROM_LINKS.xlsx")
print(course_lookup.head())

# ==========================================
# STEP 4: MERGE BACK TO YOUR MAIN FILE
# ==========================================

# 🔹 CLEAN COURSE CODE FIRST (THIS IS THE NEW PART)
df["Course_Code"] = df["Course_Code"].str.extract(r"([A-Z]{2,4}\s?\d{3})")
df["Course_Code"] = df["Course_Code"].str.strip()

course_lookup_simple = (
    course_lookup.sort_values(["Course_Code", "Course_Name"])
    .drop_duplicates(subset=["Course_Code"])
    [["Course_Code", "Course_Name"]]
)

merged = df.merge(course_lookup_simple, on="Course_Code", how="left", suffixes=("", "_lookup"))

# If existing Course_Name is messy or blank, replace with clean lookup title
if "Course_Name_lookup" in merged.columns:
    merged["Course_Name"] = merged["Course_Name_lookup"].combine_first(merged.get("Course_Name"))
    merged = merged.drop(columns=["Course_Name_lookup"])


# 🔹 SAVE USING ORIGINAL NAME (overwrite for Tableau)
merged.to_excel("PIMA_FULL_LONG_latest.xlsx", index=False)

print("✅ DONE - Updated original file for Tableau")

✔ BIO 156IN -> Intro Biology Allied Health
✔ PSY 132 -> Psychology and Culture
✔ SOC 101 -> Introduction to Sociology [SUN# SOC 1101]
✔ CHM 151IN -> General Chemistry I [SUN# CHM 1151]
✔ MAT 151 -> College Algebra [SUN# MAT 1151]
✔ MAT 189 -> Trigonometry [SUN# MAT 1187]
✔ REA 112HP -> Critical Reading for Health Professions
✔ REA 112 -> Critical Reading
✔ MUS 160 -> Popular Music in America
✔ WRT 101 -> English Composition I [SUN# ENG 1101]
✔ PSY 215 -> Human Sexuality
✔ BIO 201IN -> Human Anatomy and Physiology I [SUN# BIO 2201]
✔ BIO 201IH -> Human Anatomy, Physiology and Histology
✔ BIO 205IN -> Microbiology [SUN# BIO 2205]
✔ GTW 101 -> Writing for Trades and Technical Occupations
✔ GTM 105 -> Applied Technical Mathematics
✔ MAT 106 -> Elementary Data Analysis with Spreadsheets
✔ EDC 266 -> Internship Practicum
✔ EDC 267 -> Traditional Practicum
✔ BIO 160IN -> Introduction to Human Anatomy and Physiology
✔ PSY 101 -> Introduction to Psychology [SUN# PSY 1101]
✔ THE 105 -> Theater A

# SOC Economic Master File (Detailed Version)

## Script
Projections + OEWS merge with strict SOC cleaning and formatting

## Purpose
Create a finalized occupation-level dataset with employment, wages, openings, and growth for Arizona

## What it does

### Loads projections data
- Reads projections Excel file (ltop dataset)
- Uses header row offset (header=2)
- Cleans column names (removes spaces, line breaks)

### Standardizes columns
- Renames:
  - Area Name → Area
  - SOC Code2 → SOC
  - Occupation Title → Occupation
- Filters to Arizona only

### Cleans SOC codes
- Removes whitespace and formatting issues
- Removes decimals (.00, .0)
- Ensures consistent SOC format

### Removes aggregate SOCs (early step)
- Keeps only valid SOC format (##-####)
- Drops SOC codes ending in "0000"

### Builds core labor market metrics
- Employment → from 2024 Estimates
- Annual Openings → Total7 divided by 10
- 10-Year Growth (%) → cleaned numeric values

### Creates base dataset
- Keeps:
  - SOC
  - Occupation
  - Employment
  - Annual Openings
  - 10-Year Growth (%)

### Loads OEWS wage data
- Reads OEWS Excel file
- Cleans column names
- Cleans SOC codes
- Selects 50th percentile wage
- Renames to Median Wage

### Merges wage data
- Left join on SOC
- Adds wage information to master dataset

### Formats dataset
- Keeps key columns:
  - SOC
  - Occupation
  - Median Wage
  - Employment
  - Annual Openings
  - 10-Year Growth (%)
- Sorts by Annual Openings (descending)

### Removes aggregate SOCs again (final validation)
- Ensures only detailed occupations remain
- Filters out any SOC ending in "0000"

### Final transformations
- Converts growth to percentage (×100)
- Converts hourly wage to annual wage (×2080)
- Renames to Median Annual Wage
- Drops original Median Wage column

### Final structure
- SOC
- Occupation
- Median Annual Wage
- Employment
- Annual Openings
- 10-Year Growth (%)

### Exports final dataset
- Saves to MASTER_SOC_ECONOMIC_FILE.xlsx
- Prints confirmation and preview

## Key Notes
- Aggregate SOC removal is performed multiple times for safety
- Annual openings are derived from 10-year totals
- Wage conversion assumes full-time (2080 hours/year)
- This dataset is used for Tableau dashboards and pathway analysis

In [17]:
import pandas as pd
import numpy as np
import re

# ======================================================
# FILE PATHS
# ======================================================

PATH_PROJ = r"C:\Users\301533\Downloads\ltop-04-2024to2034 (1).xlsx"
PATH_OEWS = r"C:\Users\301533\Documents\Green Analysis\Arizona OEWS 2024.xlsx"

OUT_PATH = r"C:\Users\301533\Documents\AI Analysis\MASTER_SOC_ECONOMIC_FILE.xlsx"

# ======================================================
# HELPER FUNCTIONS
# ======================================================

def clean_soc(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", "", s)
    s = s.replace(".00","").replace(".0","")
    return s

def to_num(series):
    return (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

# ======================================================
# LOAD PROJECTIONS (MAIN DATA SOURCE)
# ======================================================

proj = pd.read_excel(PATH_PROJ, header=2)

# Clean column names
proj.columns = (
    proj.columns.astype(str)
    .str.strip()
    .str.replace("\n"," ", regex=False)
    .str.replace(r"\s+"," ", regex=True)
)

# Rename key columns
proj = proj.rename(columns={
    "Area Name": "Area",
    "SOC Code2": "SOC",
    "Occupation Title": "Occupation"
})

# Keep Arizona only
proj = proj[proj["Area"] == "Arizona"].copy()

# Clean SOC codes
proj["SOC"] = proj["SOC"].apply(clean_soc)

# ======================================================
# REMOVE AGGREGATE SOCs EARLY (CRITICAL FIX)
# ======================================================

proj["SOC"] = proj["SOC"].astype(str)

proj = proj[
    proj["SOC"].str.match(r"\d{2}-\d{4}") & 
    ~proj["SOC"].str.endswith("0000")
]

# ======================================================
# BUILD CORE METRICS
# ======================================================

# Employment (CURRENT jobs)
proj["Employment"] = pd.to_numeric(
    to_num(proj["2024 Estimates3"]),
    errors="coerce"
)

# Annual Openings (Total7 = 10-year total → divide by 10)
proj["Annual Openings"] = pd.to_numeric(
    to_num(proj["Total7"]),
    errors="coerce"
) / 10

# 10-Year Growth %
proj["10-Year Growth (%)"] = pd.to_numeric(
    to_num(proj["Percentage Change"]),
    errors="coerce"
)

# Keep only needed columns
master = proj[[
    "SOC",
    "Occupation",
    "Employment",
    "Annual Openings",
    "10-Year Growth (%)"
]].copy()

# ======================================================
# LOAD OEWS (WAGES)
# ======================================================

oews = pd.read_excel(PATH_OEWS, skiprows=1)

oews.columns = oews.columns.str.strip()

oews["SOC"] = oews["SOC Code1"].apply(clean_soc)

oews_use = oews[[
    "SOC",
    "50th Percentile Wage"
]].rename(columns={
    "50th Percentile Wage": "Median Wage"
})

# Merge wages into master
master = master.merge(
    oews_use,
    on="SOC",
    how="left"
)

# ======================================================
# FINAL FORMAT
# ======================================================

master = master[[
    "SOC",
    "Occupation",
    "Median Wage",
    "Employment",
    "Annual Openings",
    "10-Year Growth (%)"
]]

# Sort by most opportunity
master = master.sort_values("Annual Openings", ascending=False)

# ======================================================
# REMOVE AGGREGATE SOCs (FINAL FIX)
# ======================================================

master["SOC"] = master["SOC"].astype(str)

master = master[
    master["SOC"].str.match(r"\d{2}-\d{4}") &   # valid SOC format
    ~master["SOC"].str.endswith("0000")         # remove aggregates
]
# ======================================================
# CLEAN FINAL DATA
# ======================================================

# Ensure SOC is string
master["SOC"] = master["SOC"].astype(str)

# Remove aggregate SOCs
master = master[
    master["SOC"].str.match(r"\d{2}-\d{4}") &
    ~master["SOC"].str.endswith("0000")
]

# Convert growth to %
master["10-Year Growth (%)"] = master["10-Year Growth (%)"] * 100

# Convert wage to annual
master["Median Annual Wage"] = master["Median Wage"] * 2080
master = master.drop(columns=["Median Wage"])

# Reorder
master = master[[
    "SOC",
    "Occupation",
    "Median Annual Wage",
    "Employment",
    "Annual Openings",
    "10-Year Growth (%)"
]]

master = master[~master["SOC"].str.endswith("0000")]

# ======================================================
# EXPORT
# ======================================================
# Remove aggregate SOCs (ONLY keep detailed occupations)
master = master[~master["SOC"].astype(str).str.endswith("0000")]

master.to_excel(OUT_PATH, index=False)

print("✅ MASTER FILE CREATED:")
print(OUT_PATH)
print(master.head())

✅ MASTER FILE CREATED:
C:\Users\301533\Documents\AI Analysis\MASTER_SOC_ECONOMIC_FILE.xlsx
         SOC                         Occupation Median Annual Wage  \
443  35-3000  Food and Beverage Serving Workers                NaN   
506  41-2000               Retail Sales Workers                NaN   
830  53-7000            Material Moving Workers                NaN   
445  35-3023      Fast Food and Counter Workers            33051.2   
543  43-4000      Information and Record Clerks                NaN   

     Employment  Annual Openings  10-Year Growth (%)  
443    161484.0          38577.8               14.89  
506    176511.0          30504.2                7.69  
830    158801.0          24929.4               14.72  
445     86839.0          22870.8               17.97  
543    152296.0          19778.5                2.54  


# SOC Economic Master File

## Script
Projections + OEWS merge (Arizona workforce data)

## Purpose
Create a master dataset of occupations with employment, wages, openings, and growth

## What it does

### Loads projections data
- Reads projections Excel file (ltop dataset)
- Uses correct header row (header=2)
- Cleans column names (removes spaces, line breaks)

### Standardizes core fields
- Renames columns:
  - Area Name → Area
  - SOC Code2 → SOC
  - Occupation Title → Occupation
- Filters to Arizona only

### Cleans SOC codes
- Removes spaces and decimals (.00, .0)
- Ensures consistent SOC format

### Removes aggregate occupations (critical step)
- Drops SOC codes ending in "0000"
- Keeps only detailed occupations

### Builds core labor market metrics
- Employment → from 2024 Estimates
- Annual Openings → Total7 divided by 10
- 10-Year Growth (%) → cleaned and converted to numeric

### Creates base dataset
- Keeps:
  - SOC
  - Occupation
  - Employment
  - Annual Openings
  - 10-Year Growth (%)

### Loads wage data (OEWS)
- Reads OEWS Excel file
- Cleans SOC codes
- Selects median wage (50th percentile)
- Renames to "Median Wage"

### Merges wage data
- Left join on SOC
- Adds wage information to dataset

### Final cleaning and transformations
- Removes any remaining aggregate SOCs
- Converts growth to percentage (×100)
- Converts hourly wage → annual wage (×2080)
- Renames to "Median Annual Wage"

### Final formatting
- Reorders columns:
  - SOC
  - Occupation
  - Median Annual Wage
  - Employment
  - Annual Openings
  - 10-Year Growth (%)
- Sorts by Annual Openings (descending)

### Exports final dataset
- Saves to MASTER_SOC_ECONOMIC_FILE.xlsx
- Prints preview of results

## Key Notes
- Removing SOC codes ending in "0000" is essential
- Annual openings are derived from 10-year totals
- Wage conversion assumes full-time hours (2080/year)
- This is the core dataset for Tableau and pathway analysis

In [18]:
import pandas as pd
import numpy as np
import re

# ======================================================
# FILE PATHS
# ======================================================

PATH_PROJ = r"C:\Users\301533\Downloads\ltop-04-2024to2034 (1).xlsx"
PATH_OEWS = r"C:\Users\301533\Documents\Green Analysis\Arizona OEWS 2024.xlsx"

OUT_PATH = r"C:\Users\301533\Documents\AI Analysis\MASTER_SOC_ECONOMIC_FILE.xlsx"

# ======================================================
# HELPER FUNCTIONS
# ======================================================

def clean_soc(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", "", s)
    s = s.replace(".00","").replace(".0","")
    return s

def to_num(series):
    return (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
    )

# ======================================================
# LOAD PROJECTIONS
# ======================================================

proj = pd.read_excel(PATH_PROJ, header=2)

proj.columns = (
    proj.columns.astype(str)
    .str.strip()
    .str.replace("\n"," ", regex=False)
    .str.replace(r"\s+"," ", regex=True)
)

proj = proj.rename(columns={
    "Area Name": "Area",
    "SOC Code2": "SOC",
    "Occupation Title": "Occupation"
})

# Keep Arizona only
proj = proj[proj["Area"] == "Arizona"].copy()

# Clean SOC
proj["SOC"] = proj["SOC"].apply(clean_soc).astype(str).str.strip()

# ======================================================
# REMOVE AGGREGATE SOCs EARLY (KEY STEP)
# ======================================================

proj = proj[proj["SOC"].str[-4:] != "0000"]

# ======================================================
# BUILD METRICS
# ======================================================

proj["Employment"] = pd.to_numeric(
    to_num(proj["2024 Estimates3"]),
    errors="coerce"
)

proj["Annual Openings"] = pd.to_numeric(
    to_num(proj["Total7"]),
    errors="coerce"
) / 10

proj["10-Year Growth (%)"] = pd.to_numeric(
    to_num(proj["Percentage Change"]),
    errors="coerce"
)

# Keep needed columns
master = proj[[
    "SOC",
    "Occupation",
    "Employment",
    "Annual Openings",
    "10-Year Growth (%)"
]].copy()

# ======================================================
# LOAD OEWS (WAGES)
# ======================================================

oews = pd.read_excel(PATH_OEWS, skiprows=1)
oews.columns = oews.columns.str.strip()

oews["SOC"] = oews["SOC Code1"].apply(clean_soc).astype(str).str.strip()

oews_use = oews[[
    "SOC",
    "50th Percentile Wage"
]].rename(columns={
    "50th Percentile Wage": "Median Wage"
})

# Merge wages
master = master.merge(oews_use, on="SOC", how="left")

# ======================================================
# FINAL CLEANING
# ======================================================

# Remove any leftover bad SOCs (extra safety)
master["SOC"] = master["SOC"].astype(str).str.strip()
master = master[master["SOC"].str[-4:] != "0000"]

# Convert growth to %
master["10-Year Growth (%)"] = master["10-Year Growth (%)"] * 100

# Convert hourly wage → annual
master["Median Annual Wage"] = master["Median Wage"] * 2080
master = master.drop(columns=["Median Wage"])

# Final column order
master = master[[
    "SOC",
    "Occupation",
    "Median Annual Wage",
    "Employment",
    "Annual Openings",
    "10-Year Growth (%)"
]]

# Sort
master = master.sort_values("Annual Openings", ascending=False)

# ======================================================
# EXPORT
# ======================================================

master.to_excel(OUT_PATH, index=False)

print("✅ MASTER FILE CREATED:")
print(OUT_PATH)
print(master.head())

✅ MASTER FILE CREATED:
C:\Users\301533\Documents\AI Analysis\MASTER_SOC_ECONOMIC_FILE.xlsx
         SOC                         Occupation Median Annual Wage  \
443  35-3000  Food and Beverage Serving Workers                NaN   
506  41-2000               Retail Sales Workers                NaN   
830  53-7000            Material Moving Workers                NaN   
445  35-3023      Fast Food and Counter Workers            33051.2   
543  43-4000      Information and Record Clerks                NaN   

     Employment  Annual Openings  10-Year Growth (%)  
443    161484.0          38577.8               14.89  
506    176511.0          30504.2                7.69  
830    158801.0          24929.4               14.72  
445     86839.0          22870.8               17.97  
543    152296.0          19778.5                2.54  


# PIMA Clean Tables Export

## Script
Program and course table extraction for Tableau

## Purpose
Create clean program-level and course-level tables from the full dataset

## What it does

### Loads input dataset
- Reads Excel file: PIMA_FULL_LONG_latest.xlsx
- Stores data in a DataFrame

### Cleans column names
- Removes leading and trailing spaces from all column names

### Creates program-level table
- Selects:
  - Program
  - Credits
  - Duration_Weeks
- Removes duplicate rows
- Sorts alphabetically by Program

### Creates course-level table
- Selects:
  - Program
  - Course_Code
  - Course_Name
- Removes duplicate rows
- Sorts by Program and Course_Name

### Exports to Excel
- Creates a new Excel file: PIMA_CLEAN_TABLES.xlsx
- Writes two sheets:
  - Programs (program-level data)
  - Courses (course-level data)

### Saves output
- Uses ExcelWriter with openpyxl engine
- Saves both tables into one file

## Key Notes
- Removes duplicates to ensure clean relational tables
- Designed for Tableau data modeling (separate tables)
- Sorting improves readability and usability in dashboards

In [2]:
import pandas as pd

# =========================
# 1. LOAD YOUR DATA
# =========================
input_file = r"C:\Users\301533\Documents\GitHub\PCIC\analysis\PIMA_FULL_LONG_latest.xlsx"

df = pd.read_excel(input_file)

# =========================
# 2. CLEAN COLUMN NAMES
# =========================
df.columns = df.columns.str.strip()

# =========================
# 3. CREATE PROGRAM-LEVEL TABLE
# =========================
programs = df[['Program', 'Credits', 'Duration_Weeks']].drop_duplicates()

# Sort (optional)
programs = programs.sort_values(by='Program')

# =========================
# 4. CREATE COURSE-LEVEL TABLE
# =========================
courses = df[['Program', 'Course_Code', 'Course_Name']].drop_duplicates()

# Sort (optional)
courses = courses.sort_values(by=['Program', 'Course_Name'])

# =========================
# 5. SAVE TO EXCEL
# =========================
output_file = r"C:\Users\301533\Documents\GitHub\PCIC\analysis\PIMA_CLEAN_TABLES.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    programs.to_excel(writer, sheet_name='Programs', index=False)
    courses.to_excel(writer, sheet_name='Courses', index=False)

print("✅ DONE! File saved to:")
print(output_file)

✅ DONE! File saved to:
C:\Users\301533\Documents\GitHub\PCIC\analysis\PIMA_CLEAN_TABLES.xlsx
